# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset on rangeland management practices in Northern Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset schema is described by a Croissant schema JSON-LD file accessible at:
- https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR^2 metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print basic dataset metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, columns, and their `@id` identifiers so that each is referenced unambiguously in further steps.

In [ ]:
# Helper: print record sets and their fields with @ids
def list_record_sets_and_fields(metadata):
    print("Available Record Sets:")
    if not hasattr(metadata, 'record_sets') or not metadata.record_sets:
        print('No record sets found in the Croissant schema.')
        return []
    record_sets = metadata.record_sets
    for rs in record_sets:
        print(f"- Record Set name: '{getattr(rs, 'name', 'N/A')}', @id: '{rs.id}'")
        print("  Fields:")
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"    - {getattr(field, 'name', 'N/A')} (@id: {field.id})")
        print()
    return [rs.id for rs in record_sets]

record_set_ids = list_record_sets_and_fields(metadata)

## 3. Data Extraction
Load one or more record sets into Pandas DataFrames for exploration. All entities (record sets, fields, and columns) are referenced by their `@id` only.

In [ ]:
# If no record sets are printed above, the following will be demonstration only.
if not record_set_ids:
    print("No record sets defined in the metadata. Check dataset schema or contact the provider.")
else:
    dataframes = {}
    for record_set_id in record_set_ids:
        print(f"Loading records for record set: {record_set_id}")
        records_iter = dataset.records(record_set=record_set_id)
        try:
            records = list(records_iter)
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded DataFrame for record set {record_set_id} with columns: {df.columns.tolist()}")
                print(df.head(2))
            else:
                print(f"No records found for record set {record_set_id}.")
        except Exception as e:
            print(f"Error loading data for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Process or filter the records from a chosen record set. All field and record set references must be by their `@id` (not field name strings).

In [ ]:
# Choose a record set @id for EDA (choose first if possible)
if not record_set_ids or not dataframes:
    print("Cannot proceed with EDA: No dataframes were loaded.")
else:
    # Use the first record set for demonstration
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Select the first numeric field @id available (fallback to the first column)
    numeric_field_id = None
    group_field_id = None
    if hasattr(metadata, 'record_sets'):
        for rs in metadata.record_sets:
            if rs.id == record_set_id and hasattr(rs, 'fields'):
                for f in rs.fields:
                    # Heuristic: treat any field with type integer or float as numeric
                    if hasattr(f, 'data_type') and f.data_type in ('schema:Integer', 'schema:Float', 'schema:Number'):
                        numeric_field_id = f.id
                    # Heuristic: use first non-numeric field as group
                    if hasattr(f, 'data_type') and f.data_type == 'schema:Text' and not group_field_id:
                        group_field_id = f.id
                break
    # Fallback if numeric/group fields informative lookup fails
    if numeric_field_id is None and not df.empty:
        # Try to infer first numeric column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if group_field_id is None and not df.empty and len(df.columns) > 1:
        group_field_id = df.columns[1]

    if numeric_field_id is None:
        print("No numeric field found for analysis.")
    else:
        threshold = 10
        print(f"Filtering DataFrame on field '@id': {numeric_field_id} > {threshold}")
        # For demonstration, ensure field is numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize field distributions or relationships from the dataset using pandas and matplotlib. (Fields referenced **by `@id`**.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not dataframes or numeric_field_id is None:
    print('Nothing to plot: No suitable numeric field loaded.')
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}' in record set '{record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook showed the process of loading and exploring a FAIR^2-compliant dataset using the `mlcroissant` library. Record sets, fields, and columns were referenced **solely by their `@id`s**. We loaded available data, performed basic analysis and filtering on numeric fields, grouped and visualized variables, and prepared the data for further downstream tasks. For more advanced operations, refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) or dataset-specific codebooks.

**Key takeaways:**
- All dataset entities were referenced by their `@id` for clarity and reproducibility.
- The notebook can be adapted for any Croissant-compatible dataset by updating the Croissant schema URL.
- Further wrangling may require detailed inspection of field schemas and types, which is straightforward with mlcroissant's metadata model.